## K-means로 비슷한 표본 묶기


### 패키지 설치

터미널에서 설치한다.

```bash
pip install plotly
```


In [2]:
import plotly.express as px
import pandas as pd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

SEED = 42


### 군집화

군집화(clustering)는 정답 레이블 없이 입력 특성이 비슷한 표본을 묶는 비지도학습 방법이다.
군집 번호는 그룹을 구별하는 이름이다. 군집별 표본 수로 그룹의 크기를 확인하고, 특성별 평균으로 각 그룹의 특징을 살펴볼 수 있다.

### K-means

K-means는 표본과 중심 사이의 거리를 이용해 K개 군집을 만든다.
중심점은 각 군집에 속한 표본의 특성별 평균이다.

초기 중심을 정한 뒤 표본을 가장 가까운 중심에 배정하고, 군집별 평균으로 중심을 갱신한다.
중심 위치가 거의 변하지 않을 때까지 배정과 갱신을 반복한다.


### 붓꽃의 크기로 표본 묶기

꽃받침과 꽃잎의 크기가 비슷한 붓꽃을 묶는 문제다.
K-means로 네 측정값이 가까운 표본을 군집으로 만들고, 군집별 평균으로 그룹의 특징을 설명한다.

### 붓꽃 데이터 불러오기

Iris에는 붓꽃 150개 표본의 실제 측정값이 있다. 한 행은 붓꽃 표본 하나다.
꽃받침(sepal)은 꽃잎(petal) 바깥쪽에서 꽃을 받치는 부분이다.

| 역할 | 열 | 의미와 단위 |
| --- | --- | --- |
| 입력 | `sepal length (cm)` | 꽃받침 길이, cm |
| 입력 | `sepal width (cm)` | 꽃받침 너비, cm |
| 입력 | `petal length (cm)` | 꽃잎 길이, cm |
| 입력 | `petal width (cm)` | 꽃잎 너비, cm |

`load_iris`는 scikit-learn에 포함된 Iris 데이터를 불러오는 함수다.
붓꽃 150개의 측정값을 입력 X로 사용해 비슷한 표본을 묶는다. 종 정보는 학습에 사용하지 않는다.


In [3]:
iris = load_iris(as_frame=True)
X = iris.data
print("전체 데이터:", X.shape)
display(X.head())


전체 데이터: (150, 4)


,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2


### 입력 특성 표준화

특성별 값의 퍼짐이 거리 계산에 미치는 영향을 조정하기 위해 표준화한다.


In [4]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


### K-means 학습과 시각화

`KMeans`로 표준화한 네 특성을 세 군집으로 묶는다.
K는 사용자가 정하는 군집 수다.

색은 군집 번호, X 표시는 중심이다.
군집은 네 특성으로 학습했으며, 그래프에는 꽃잎의 길이와 너비를 cm 단위로 표시한다.


In [6]:
# n_clusters: 군집 수, n_init: 초기 중심을 바꿔 학습하는 횟수
model = KMeans(n_clusters=3, n_init=10, random_state=SEED)
model.fit(X_scaled)

# cluster_centers_: 군집 중심 (군집 수, 4). inverse_transform으로 cm 단위 복원
centers = pd.DataFrame(
    scaler.inverse_transform(model.cluster_centers_), columns=X.columns
)
# labels_: 표본별 군집 번호. 문자열로 바꾸어 색을 범주별로 구분한다.
# concat: 표본과 중심을 행으로 합쳐 한 그림에 표시한다.
plot_data = pd.concat([
    X.assign(cluster=model.labels_.astype(str), kind="sample"),
    centers.assign(cluster=centers.index.astype(str), kind="center"),
], ignore_index=True)

# 두 군집 그래프에서 같은 군집에 같은 색을 사용한다.
cluster_colors = dict(zip(plot_data["cluster"].unique(), px.colors.qualitative.Plotly))

# scatter: 열 이름으로 축·색·모양을 지정하고 Figure를 반환한다.
fig = px.scatter(
    plot_data, x="petal length (cm)", y="petal width (cm)",
    color="cluster", color_discrete_map=cluster_colors,
    symbol="kind", symbol_map={"sample": "circle", "center": "x"},
    title=f"K-means: K={model.n_clusters}",
)
fig.update_traces(marker_size=12, selector={"marker.symbol": "x"})  # 중심 크기
fig.show()

### 세 특성으로 군집 보기

같은 군집을 꽃받침 길이·꽃잎 길이·꽃잎 너비의 세 축으로 표시한다.
색과 X 표시는 각각 군집 번호와 중심이며, 축의 단위는 cm다.
마우스로 드래그해 회전하고, 휠로 확대·축소할 수 있다.


In [10]:
# scatter_3d: 세 열을 x·y·z축으로 사용하는 3차원 산점도
fig = px.scatter_3d(
    plot_data, x="sepal length (cm)", y="petal length (cm)", z="petal width (cm)",
    color="cluster", color_discrete_map=cluster_colors,
    symbol="kind", symbol_map={"sample": "circle", "center": "x"},
    title=f"K-means: K={model.n_clusters}",
)
fig.update_traces(marker_size=3, selector={"marker.symbol": "circle"})
fig.update_traces(marker_size=6, selector={"marker.symbol": "x"})
fig.show()


### 군집별 특징 살펴보기

군집별 표본 수와 네 측정값의 평균을 비교한다.
꽃잎이 짧고 좁은 군집과 길고 넓은 군집을 찾아 그래프와 연결해 설명해 보자.


In [11]:
clustered = X.assign(cluster=model.labels_)
profile = clustered.groupby("cluster")[X.columns].mean()
profile.insert(0, "count", clustered.groupby("cluster").size())
display(profile.round(2))


,count,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm)
cluster,,,,,
0,53,5.80,2.67,4.37,1.41
1,50,5.01,3.43,1.46,0.25
2,47,6.78,3.10,5.51,1.97


### 실제 종과 비교하기

학습에 사용하지 않은 종 정보를 가져와 K=3 군집과 비교한다.
교차표의 행은 군집 번호, 열은 실제 종이며, 각 칸은 해당 표본 수다.


In [12]:
# target: 표본별 종 번호 Series (150,), target_names: 번호 순서의 종 이름
species = iris.target.map(dict(enumerate(iris.target_names)))
clusters = pd.Series(model.labels_, index=X.index, name="cluster")
# crosstab: 군집과 종의 조합별 표본 수를 DataFrame으로 반환
comparison = pd.crosstab(clusters, species.rename("species"))
display(comparison)


species,setosa,versicolor,virginica
cluster,,,
0,0,39,14
1,50,0,0
2,0,11,36


### 실제 종의 분포

같은 세 축에 실제 종을 색으로 표시한다.
각 종은 교차표에서 해당 종이 가장 많이 속한 군집과 같은 색을 사용한다.
위의 군집 그림과 비교해 versicolor와 virginica가 섞이는 영역을 살펴보자.


In [13]:
# idxmax: 각 종의 표본 수가 가장 많은 군집 번호를 반환한다.
species_colors = {
    name: cluster_colors[str(cluster)]
    for name, cluster in comparison.idxmax(axis=0).items()
}
fig = px.scatter_3d(
    X.assign(species=species),
    x="sepal length (cm)", y="petal length (cm)", z="petal width (cm)",
    color="species", color_discrete_map=species_colors,
    title="Iris species",
)
fig.update_traces(marker_size=3)
fig.show()
